# Multimodal AI — Blood Work Analysis

In [9]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent
import base64

load_dotenv()

True

## Encode the image and send to the vision model

In [ ]:
with open("medical_report.png", "rb") as f:
    image_bytes = f.read()
    image_b64 = base64.b64encode(image_bytes).decode("utf-8")

## extract the information with help of llm model 

In [5]:
llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")

message = HumanMessage(content=[
    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
    {"type": "text",      "text": "This is a blood work report. Extract all test results and flag any values outside the normal range."}
])

response = llm.invoke([message])
print(response.content)

The attached image shows a complete blood count test result. The results are as follows:

**Complete Blood Count**

*   **Test**                         **Result**   **Normal Range**
*   Haemoglobin (Hb)                15          14 - 16 g%
*   RBC Count                       5           4.32 - 5.72 million/μL
*   PCV                             35          35 - 45 %
*   MCV                             72.00       80 - 99 fL
*   MCH                             20.00       26 - 32 pg
*   MCHC                            41.67       30 - 34 %
*   RDW                             10           9 - 17 %
*   **Total WBC Count**             **5500**     **4000 - 11000 /cu.mm**
*   *Neutrophils*                   80          40 - 75 %
*   *Lymphocytes*                   50          25 - 45 %
*   *Eosinophils*                   5           00 - 06 %
*   *Monocytes*                     5           00 - 10 %
*   *Basophils*                     0           00 - 01 %
*   **Platelet Count**          

## Suggest Diet Plan Agent

In [6]:
@tool
def get_diet_recommendation(condition: str) -> dict:
    """Given a health condition, returns a diet plan. Condition must be one of: normal, high_cholesterol, high_sugar."""
    diet_plans = {
        "high_cholesterol": {
            "eat":        ["fruits", "vegetables", "whole grains", "lean protein"],
            "do_not_eat": ["red meat", "fried food", "full-fat dairy", "processed snacks"],
        },
        "high_sugar": {
            "eat":        ["vegetables", "whole grains", "legumes", "nuts"],
            "do_not_eat": ["white rice", "white sugar", "junk food", "sugary drinks"],
        },
        "normal": {
            "eat":        ["vegetables", "fruits", "whole grains", "lean protein"],
            "do_not_eat": ["excessive sugar", "processed food", "trans fats"],
        },
    }
    return diet_plans.get(condition, diet_plans["normal"])

In [7]:
SYSTEM_PROMPT = """
You are a helpful medical and nutrition assistant.
For the input blood work image, extract the numbers and the normal range, then categorize
the condition as one of: normal, high_cholesterol, high_sugar.
Then call the appropriate tool to retrieve and present the diet plan.
"""

diet_agent = create_agent(
    llm,
    tools=[get_diet_recommendation],
    system_prompt=SYSTEM_PROMPT,
)

In [10]:
result = diet_agent.invoke({
    "messages": [HumanMessage(content=[
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
        {"type": "text",      "text": "Analyse this blood work report and suggest a diet plan."},
    ])]
})

print(result["messages"][-1].content)

The patient's blood work report appears to be within normal limits. The diet plan for a normal condition typically includes a balanced intake of vegetables, fruits, whole grains, and lean protein sources. It is recommended to limit or avoid excessive sugar, processed foods, and trans fats.
